# 01 â€” Scenarios (compute)

Run after `00_setup.ipynb`.

This notebook is the **compute** stage. For each scenario it:
1. Rebuilds the foreground database with scenario-specific exchange amounts
2. Runs all six LCIA methods (land use, water, N/P eutrophication, climate change, freshwater ecotoxicity)
3. Saves every numeric result to `results/tables/{scenario}/...` and combined cross-scenario tables to `results/tables/`

**No plotting happens here.** All figures are produced from the saved CSVs by `02_results.ipynb`, so the plots can be iterated on without re-running the LCAs.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

_prefix = sys.prefix
if os.name == "nt":  # Windows
    os.environ.setdefault("GDAL_DATA", os.path.join(_prefix, "Library", "share", "gdal"))
    os.environ.setdefault("PROJ_LIB",  os.path.join(_prefix, "Library", "share", "proj"))
else:  # Linux / macOS
    os.environ.setdefault("GDAL_DATA", os.path.join(_prefix, "share", "gdal"))
    os.environ.setdefault("PROJ_LIB",  os.path.join(_prefix, "share", "proj"))


import bw2data as bd
import bw2calc as bc
import bw2regional as bwr
import geopandas as gpd
gpd.options.io_engine = "fiona"
import rasterstats
import numpy as np
import pandas as pd

from src.config import (
    PROJECT_NAME, DB_BIOSPHERE, DB_ECOINVENT, DB_FOREGROUND,
    TZ_DISTRICTS_PATH, WWF_ECOREGIONS_PATH, GINNERIES_PATH, TEXTILE_PLANTS_PATH,
    XT_COTTON_PRODUCTION,
    METHOD_LAND_USE_REGIONAL, METHOD_LAND_USE_GENERIC,
    METHOD_WATER, METHOD_N_EUTRO, METHOD_P_EUTRO, METHOD_CLIMATE_CHANGE,
    METHOD_ECOTOX_FW,
    RESULTS_TABLES_DIR,
)
from src.scenarios import (
    BASELINE, HIGH_YIELD, EXTENSIFICATION, ORGANIC_EXPANSION,
    MANUFACTURING_EXPANSION, IRRIGATION,
    Scenario,
)
from src.inventory import build_foreground_db

bd.projects.set_current(PROJECT_NAME)
print("Active project:", bd.projects.current)

C:\Users\elishaw\AppData\Local\miniconda3\envs\bw25-regional\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Active project:

tz_cotton

## Spatial data (load once before the scenario loop)

In [2]:
tz_districts_gdf   = gpd.read_file(TZ_DISTRICTS_PATH)
wwf_ecoregions_gdf = gpd.read_file(WWF_ECOREGIONS_PATH)
ginneries_gdf      = gpd.read_file(GINNERIES_PATH)
textile_plants_gdf = gpd.read_file(TEXTILE_PLANTS_PATH)

# Production weights from extension table
xt = bwr.ExtensionTable(XT_COTTON_PRODUCTION)
xtable_data = xt.load()
production_by_district = {district_id: value for value, (_, district_id) in xtable_data}
total_production = sum(production_by_district.values())
production_weights = {d: v / total_production for d, v in production_by_district.items()}

# Ginning weights
ginneries_proj = ginneries_gdf.to_crs(tz_districts_gdf.crs)
ginneries_by_district = gpd.sjoin(
    ginneries_proj,
    tz_districts_gdf[["ADM2_PCODE", "ADM2_EN", "geometry"]],
    how="left", predicate="within",
)
_SKIP_COLS = {"index_right", "SymbolID", "AltMode", "Base", "HasLabel", "LabelID"}
if "Production" in ginneries_by_district.columns:
    ginnery_prod_col = "Production"
else:
    _num = ginneries_by_district.select_dtypes(include=[float, int]).columns.tolist()
    _candidates = [c for c in _num if c not in _SKIP_COLS]
    ginnery_prod_col = _candidates[0] if _candidates else None

if ginnery_prod_col and ginnery_prod_col in ginneries_by_district.columns:
    agg = ginneries_by_district.groupby("ADM2_PCODE")[ginnery_prod_col].sum()
    total_gin = agg.sum()
    ginning_weights = {d: v / total_gin for d, v in agg.items() if v > 0} if total_gin > 0 else production_weights.copy()
else:
    gin_districts = ginneries_by_district["ADM2_PCODE"].dropna().unique()
    ginning_weights = {d: 1 / len(gin_districts) for d in gin_districts} if len(gin_districts) else production_weights.copy()

# Textile weights
textile_proj = textile_plants_gdf.to_crs(tz_districts_gdf.crs)
textile_by_district = gpd.sjoin(
    textile_proj,
    tz_districts_gdf[["ADM2_PCODE", "ADM2_EN", "geometry"]],
    how="left", predicate="within",
)
if len(textile_by_district) > 0 and "ADM2_PCODE" in textile_by_district.columns:
    plant_counts = textile_by_district["ADM2_PCODE"].dropna().value_counts()
    total_plants = plant_counts.sum()
    textile_weights = {d: cnt / total_plants for d, cnt in plant_counts.items()} if total_plants > 0 else production_weights.copy()
else:
    textile_weights = production_weights.copy()

spatial_data = {
    "tz_districts_gdf":  tz_districts_gdf,
    "production_weights": production_weights,
    "ginning_weights":    ginning_weights,
    "textile_weights":    textile_weights,
}
print(f"Production: {len(production_weights)} districts")
print(f"Ginning:    {len(ginning_weights)} districts")
print(f"Textile:    {len(textile_weights)} districts")


Production: 88 districts

Ginning:    22 districts

Textile:    6 districts

## Scenario list

Add or modify scenarios here before running the loop below.

In [3]:
# Scenarios to run. Edit this list or override fields with dataclasses.replace(...)
# to create custom variants. See src/scenarios.py for derivation comments.

scenarios_to_run = [
    BASELINE, HIGH_YIELD,
    EXTENSIFICATION,
    ORGANIC_EXPANSION,
    MANUFACTURING_EXPANSION,
    IRRIGATION,
]

print(f"{'Scenario':<26} {'Yield':>6s}  {'Organic':>7s}  {'DomProc':>7s}  Description")
print("-" * 130)
for s in scenarios_to_run:
    print(f"{s.name:<26} {s.yield_kg_ha:6.0f}  {s.organic_share*100:6.0f}%  "
          f"{s.employment.domestic_processing_share*100:6.0f}%  {s.description}")

Scenario                    Yield  Organic  DomProc  Description

----------------------------------------------------------------------------------------------------------------------------------

baseline                      634      40%     100%  Tanzania cotton current-state baseline — yield 633.69 kg/ha (TCB 2024 calculated), 40% organic share, 20% of lint processed domestically into textile.

high_yield                   2471      40%     100%  Yield rises to the updated national target of 2,471 kg/ha (TCDS); organic share unchanged at 40%. Per-ha fertilizer/pesticide rates scale linearly with yield, so per-kg LCA inputs match BASELINE. National production rises to ~1,101,614 t/yr; mill capacity unchanged so the extra lint is exported. Higher yield drives 3.90× per-ha labour demand (~468 person-days).

extensification               634      40%     100%  Area-expansion counterfactual to HIGH_YIELD: national output matched to the TCDS target (~1.10 Mt) by expanding cultivated area 3.90x at unchanged baseline yield (633.69 kg/ha) and unchanged per-hectare input intensity. Per-kg field rates therefore equal baseline; only land requirement differs.

organic_expansion             634      80%     100%  Organic share 40% → 80% via policy push; yield held at 633.69 kg/ha (no yield penalty per Meatu field trials, Springer 2020). Synthetic pesticide use drops to ~30% of baseline.

manufacturing_expansion       634      40%     100%  Domestic textile output triples (17 → 51 kt/yr) via mill expansion; yield and organic share held at baseline values. Industrial-policy lever — sets domestic_textile_t_override directly. Observed processing share rises from 0.20 to 0.60 as a consequence, but the share is not the control.

irrigation                   1430      40%     100%  Full supplemental irrigation raises yield to 1,430 kg/ha (India national average irrigated cotton yield; ~2.26× Tanzania baseline). Per-ha fertilizer and pesticide FIXED at baseline levels — irrigation is the sole driver of yield gain, so per-kg agrochemical burdens fall by 56%. Labour scales with yield (270.84 person-days/ha). Irrigation water 3.402 m³/kg (WFN Report 68).

## Run all scenarios

In [4]:
from src.plotting import (
    compute_dls_coverage_from_scenario,
    compute_stage_scores, compute_district_scores,
    contribution_analysis,
)

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 01_scenarios is the COMPUTE notebook â€” it runs all LCAs and saves every
# numeric output to disk. All plotting lives in 02_results.ipynb so that
# figures can be iterated on without re-running the LCAs.
#
# Output layout
# -------------
#   results/tables/{scenario}/scores.csv
#   results/tables/{scenario}/stage_scores.csv
#   results/tables/{scenario}/contribution_{cat_key}.csv      (one per category)
#   results/tables/all_scenarios.csv                          (combined)
#   results/tables/dls_coverage_all_scenarios.csv             (combined)
#   results/tables/stage_scores_all_scenarios.csv             (combined, long)
#   results/tables/district_scores_all_scenarios.csv          (combined, long)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

all_results       = []
all_dls_rows      = []
all_stage_rows    = []
all_district_rows = []

# Display labels written into CSVs so 02_results can pretty-print scenarios.
SCENARIO_DISPLAY = {
    "baseline":                 "Baseline",
    "high_yield":               "High Yield",
    "extensification":             "Extensification",
    "organic_expansion":        "Organic Expansion",
    "manufacturing_expansion":  "Manufacturing Expansion",
    "irrigation":              "Irrigation",
}

# Spatial cat key -> (cat_label_in_scores_csv, lca_obj_var_name)
# (lca_obj resolved at runtime via the local variable name)
SPATIAL_CATS = {
    "land_use": "Land use occupation",
    "water":    "Water consumption",
    "n_eutro":  "FW eutrophication N",
    "p_eutro":  "FW eutrophication P",
}

for scenario in scenarios_to_run:
    print("=" * 70)
    print(f"SCENARIO: {scenario.name}")
    print(f"  {scenario.description}")
    print("=" * 70)

    # Per-scenario subfolder for all tabulated outputs
    scen_tbl_dir = os.path.join(RESULTS_TABLES_DIR, scenario.name)
    os.makedirs(scen_tbl_dir, exist_ok=True)

    # --- 1. Build foreground DB -----------------------------------------------
    build_foreground_db(scenario, spatial_data)

    # --- 2. Demand vector -----------------------------------------------------
    foreground = bd.Database(DB_FOREGROUND)
    domestic_consumption = [a for a in foreground if a["code"] == "prod8"][0]
    foreign_consumption  = [a for a in foreground if a["code"] == "prod7"][0]

    domestic_total_t = scenario.domestic_total_t()
    export_lint_t    = scenario.export_lint_t()
    print(f"  Demand: domestic basket = {domestic_total_t:>10,.0f} t/yr  "
          f"(cake={scenario.domestic_seed_cake_t():,.0f} | "
          f"oil={scenario.domestic_seed_oil_t():,.0f} | "
          f"textile={scenario.domestic_textile_t():,.0f})")
    print(f"          export lint     = {export_lint_t:>10,.0f} t/yr")

    demand_combined = {
        domestic_consumption.id: domestic_total_t,
        foreign_consumption.id:  export_lint_t,
    }

    # --- 3. Regionalized land use LCA -----------------------------------------
    lca = bwr.OneSpatialScaleLCA(demand=demand_combined, method=METHOD_LAND_USE_REGIONAL)
    lca.lci(); lca.lcia()
    print(f"Land use (regionalized):  {lca.score:.4e}  PDF*m2*yr")

    # --- 4. Site-generic land use (benchmark) ---------------------------------
    lca_sg = bc.LCA(demand=demand_combined, method=METHOD_LAND_USE_GENERIC)
    lca_sg.lci(); lca_sg.lcia()
    ratio = lca.score / lca_sg.score if lca_sg.score else float("nan")
    print(f"Land use (site-generic):  {lca_sg.score:.4e}  PDF*m2*yr  (ratio={ratio:.3f})")

    # --- 5. Water consumption -------------------------------------------------
    lca_water = bwr.OneSpatialScaleLCA(demand=demand_combined, method=METHOD_WATER)
    lca_water.lci(); lca_water.lcia()
    print(f"Water consumption:        {lca_water.score:.4e}  PDF*yr")

    # --- 6. N eutrophication --------------------------------------------------
    lca_n = bwr.OneSpatialScaleLCA(demand=demand_combined, method=METHOD_N_EUTRO)
    lca_n.lci(); lca_n.lcia()
    print(f"FW eutrophication -- N:   {lca_n.score:.4e}  PDF*yr")

    # --- 7. P eutrophication --------------------------------------------------
    lca_p = bwr.OneSpatialScaleLCA(demand=demand_combined, method=METHOD_P_EUTRO)
    lca_p.lci(); lca_p.lcia()
    print(f"FW eutrophication -- P:   {lca_p.score:.4e}  PDF*yr")

    # --- 8. Climate change ----------------------------------------------------
    lca_cc = bc.LCA(demand=demand_combined, method=METHOD_CLIMATE_CHANGE)
    lca_cc.lci(); lca_cc.lcia()
    print(f"Climate change (rcp26):   {lca_cc.score:.4e}  PDF*yr")

    # --- 8b. Freshwater ecotoxicity (USEtox 2.1, W6 Africa) -------------------
    # CFs were registered in 00_setup with the W6 unit-conversion already
    # applied, so lca_ecotox.score is directly in PDFÂ·yr â€” no manual scaling
    # downstream.
    lca_ecotox = bc.LCA(demand=demand_combined, method=METHOD_ECOTOX_FW)
    lca_ecotox.lci(); lca_ecotox.lcia()
    print(f"FW ecotoxicity (USEtox):  {lca_ecotox.score:.4e}  PDF*yr")

    # --- 9. Aggregate-score table --------------------------------------------
    RESULTS = [
        ("Land use occupation",  lca,        METHOD_LAND_USE_REGIONAL, "PDF*m2*yr",
         "OneSpatialScaleLCA + ecoregion CFs -> districts"),
        ("Water consumption",    lca_water,  METHOD_WATER,             "PDF*yr",
         "OneSpatialScaleLCA + PCR-GLOBWB basin CFs -> districts"),
        ("FW eutrophication N",  lca_n,      METHOD_N_EUTRO,           "PDF*yr",
         "OneSpatialScaleLCA + 0.5 deg raster CFs -> districts"),
        ("FW eutrophication P",  lca_p,      METHOD_P_EUTRO,           "PDF*yr",
         "OneSpatialScaleLCA + 0.5 deg raster CFs -> districts"),
        ("Climate change rcp26", lca_cc,     METHOD_CLIMATE_CHANGE,    "PDF*yr",
         "bc.LCA + global NaturalEarth CFs"),
        ("FW ecotoxicity",       lca_ecotox, METHOD_ECOTOX_FW,         "PDF*yr",
         "bc.LCA + USEtox 2.1 W6 Africa CFs (registered already in PDF*yr)"),
    ]

    print("\n" + "=" * 84)
    print(f"  Scenario: {scenario.name}")
    print("=" * 84)
    print(f"  {'Impact category':<28}  {'Score':>14}  {'Unit':<14}  Spatial approach")
    print("-" * 84)
    for name, lca_obj, _meth, unit, approach in RESULTS:
        print(f"  {name:<28}  {lca_obj.score:>14.4e}  {unit:<14}  {approach}")
    print("=" * 84)

    rows = []
    for name, lca_obj, _meth, unit, approach in RESULTS:
        rows.append({"scenario": scenario.name, "category": name,
                     "score": lca_obj.score, "unit": unit, "approach": approach})
    rows.append({"scenario": scenario.name, "category": "Land use (site-generic)",
                 "score": lca_sg.score, "unit": "PDF*m2*yr",
                 "approach": "bc.LCA + site-generic CF"})
    rows.append({"scenario": scenario.name, "category": "Ratio regionalized/site-generic",
                 "score": ratio, "unit": "-", "approach": ""})
    all_results.extend(rows)
    # per-scenario scores.csv  # retired 2026-09: contained in the *_all_scenarios combined table

    # --- 10. Stage breakdown per category (from LCA objects) ------------------
    stage_rows = []
    for cat_label, lca_obj, _meth, _unit, _appr in RESULTS:
        st = compute_stage_scores(lca_obj)
        for stage, score in st.items():
            r = {"scenario": scenario.name, "category": cat_label,
                 "stage": stage, "score": score}
            stage_rows.append(r)
            all_stage_rows.append(r)
    # per-scenario stage_scores.csv  # retired 2026-09: contained in the *_all_scenarios combined table

    # --- 11. District scores per spatial category ----------------------------
    spatial_lcas = {"land_use": lca, "water": lca_water,
                    "n_eutro": lca_n, "p_eutro": lca_p}
    for cat_key, lca_obj in spatial_lcas.items():
        ds = compute_district_scores(lca_obj, is_spatial=True,
                                     production_weights=production_weights,
                                     tz_districts_gdf=tz_districts_gdf)
        df_d = pd.DataFrame(
            [{"district_id": d, "score": s} for d, s in ds.items()]
        )
        # per-scenario district_scores_*.csv  # retired 2026-09: contained in the *_all_scenarios combined table
        for d, s in ds.items():
            all_district_rows.append({
                "scenario": scenario.name,
                "category": SPATIAL_CATS[cat_key],
                "cat_key":  cat_key,
                "district_id": d,
                "score": s,
            })

    # --- 12. Full activity contribution analysis (top-20 per category) -------
    ca_cats = [
        ("land_use", "Land use (regionalized)", lca),
        ("water",    "Water consumption",       lca_water),
        ("n_eutro",  "FW eutrophication N",     lca_n),
        ("p_eutro",  "FW eutrophication P",     lca_p),
        ("climate",  "Climate change (rcp26)",  lca_cc),
        ("ecotox",   "FW ecotoxicity (USEtox)", lca_ecotox),
    ]
    for cat_key, _label, lca_obj in ca_cats:
        df_ca = contribution_analysis(lca_obj, top_n=20)
        df_ca.to_csv(
            os.path.join(scen_tbl_dir, f"contribution_{cat_key}.csv"),
            index=False)

    # --- 13. DLS coverage -----------------------------------------------------
    cov = compute_dls_coverage_from_scenario(scenario)
    dls_rows_scen = []
    for indicator, frac in cov.items():
        r = {"scenario": scenario.name,
             "indicator": indicator.replace("\n", " "),
             "coverage": frac}
        dls_rows_scen.append(r)
        all_dls_rows.append(r)
    # per-scenario dls_coverage.csv  # retired 2026-09: contained in the *_all_scenarios combined table


# â”€â”€ Combined cross-scenario tables â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
os.makedirs(RESULTS_TABLES_DIR, exist_ok=True)

pd.DataFrame(all_results).to_csv(
    os.path.join(RESULTS_TABLES_DIR, "all_scenarios.csv"), index=False)

dls_df_all = pd.DataFrame(all_dls_rows)
dls_df_all.to_csv(
    os.path.join(RESULTS_TABLES_DIR, "dls_coverage_all_scenarios.csv"), index=False)

pd.DataFrame(all_stage_rows).to_csv(
    os.path.join(RESULTS_TABLES_DIR, "stage_scores_all_scenarios.csv"), index=False)

pd.DataFrame(all_district_rows).to_csv(
    os.path.join(RESULTS_TABLES_DIR, "district_scores_all_scenarios.csv"), index=False)

# Also persist the scenario display labels so 02_results can pick them up
pd.DataFrame(
    [{"scenario": k, "display": v} for k, v in SCENARIO_DISPLAY.items()]
).to_csv(
    os.path.join(RESULTS_TABLES_DIR, "scenario_display.csv"), index=False)

# DLS pivot (used by trade-off scatter and quick eyeballing)
print("\nDLS coverage pivot (rows=indicator, columns=scenario):")
dls_df_all.pivot_table(index="indicator", columns="scenario",
                       values="coverage", aggfunc="first")

print("\nAll scenarios complete. Tables saved under:")
print(f"  {RESULTS_TABLES_DIR}")
print("Run 02_results.ipynb to produce all figures from these CSVs.")

SCENARIO: baseline

  Tanzania cotton current-state baseline — yield 633.69 kg/ha (TCB 2024 calculated), 40% organic share, 20% of lint processed domestically into textile.

Cleared existing 'foreground' database.

Non-regionalized edges created.

Created 88 district cotton production activities.

...

  FW ecotoxicity                    1.1492e-05  PDF*yr          bc.LCA + USEtox 2.1 W6 Africa CFs (registered already in PDF*yr)


DLS coverage pivot (rows=indicator, columns=scenario):


All scenarios complete. Tables saved under:

  X:\Eli\projects\tz_cotton\python\results\tables

Run 02_results.ipynb to produce all figures from these CSVs.